# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [03:07<00:00, 37.56s/it]


In [4]:
len(deals)

49

In [5]:
deals[44].describe()

'Title: Tervis 24oz Stainless Steel Insulated Tumbler 3-Pack for $25 + free shipping\nDetails: That\'s $32 less than you\'d pay for this quantity in local stores. Use promo code "DEALNEWS" to score free shipping, too. Buy Now at MorningSave\nFeatures: triple walled insulation keeps drinks cold for up to 60 hours nonslip base\nURL: https://www.dealnews.com/products/Tervis/Tervis-24-oz-Stainless-Steel-Insulated-Tumbler-3-Pack/494045.html?iref=rss-c196'

In [6]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [7]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [8]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: TCL S5 58S5F 58" 4K HDR LED Smart TV for $235 + free shipping
Details: As one of today's daily deals, bag this $145 off savings. Buy Now at Best Buy
Features: 3840x2160p resolution HDR PRO+ Fire TV works with Alexa Bluetooth, WiFi, USB, Ethernet, HDMI Model: 58S5F
URL: https://www.dealnews.com/products/TCL/TCL-S5-58-S5-F-58-4-K-HDR-LED-Smart-TV/494064.html?iref=rss-

In [9]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [10]:
result = get_recommendations()

In [11]:
len(result.deals)

5

In [12]:
result.deals[1]

Deal(product_description='Experience ultimate convenience with the Vevor 2-Tube Shoe and Boot Dryer, a must-have for anyone who needs to keep their footwear warm and dry. Designed to quickly dry shoes and boots, this dryer uses dual tubes with a built-in blower that circulates warm air, effectively removing moisture and odors. Its compact design allows for easy storage, while its user-friendly operation ensures that your gear is ready to wear whenever you need it. Perfect for sports enthusiasts, workers, or everyday use, this dryer is efficient and reliable.', price=33.0, url='https://www.dealnews.com/products/Vevor/Vevor-2-Tube-Shoe-and-Boot-Dryer/494047.html?iref=rss-c142')

In [13]:
from agents.scanner_agent import ScannerAgent

In [14]:
agent = ScannerAgent()
result = agent.scan()

In [15]:
result

DealSelection(deals=[Deal(product_description='The TCL S5 58S5F is a 58-inch 4K HDR LED Smart TV that features an impressive 3840x2160p resolution and HDR PRO+ technology. It comes equipped with built-in Fire TV, allowing for seamless integration with Alexa for voice control. Connectivity options include Bluetooth, WiFi, USB, Ethernet, and multiple HDMI ports, making it versatile for various multimedia setups.', price=235.0, url='https://www.dealnews.com/products/TCL/TCL-S5-58-S5-F-58-4-K-HDR-LED-Smart-TV/494064.html?iref=rss-c142'), Deal(product_description='The Insignia F50 Series NS-55F501NA26 is a 55-inch 4K HDR LED UHD Smart TV that supports HDR10 and comes with a Fire TV experience. It features integrated Alexa voice control and three HDMI ports, providing ample connectivity for a variety of devices. With its stunning picture quality and user-friendly interface, this smart TV enhances your entertainment experience.', price=200.0, url='https://www.dealnews.com/products/Insignia/In